# 1 - Imports

In [ ]:
%reload_ext autoreload
%autoreload 2

In [51]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [52]:
import pandas as pd
import numpy as np

In [53]:
from src.utils import config, io
from src.features import selection, pruning

# 2 - Feature Selection (Independent of Target)

In [54]:
dataset = io.load_csv(config.INTERIM_DATA_DIR / 'merged_dataset.csv', index_col=0)
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.MKTP.PP.CD,...,SE.PRM.CUAT.ZS,SP.POP.DPND,SL.TLF.CACT.ZS,GE.EST,RQ.EST,IC.BRE.BI.OS,IC.BRE.BE.OS,IQ.CPA.PADM.XQ,FS.AST.PRVT.GD.ZS,FM.AST.PRVT.GD.ZS
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-1999,1999,AFG,7,MEA,MNA,IDX,LIC,NaN,NaN,NaN,...,NaN,108.686031,46.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG-2000,2000,AFG,7,MEA,MNA,IDX,LIC,3.521418e+09,NaN,1.637703e+10,...,NaN,109.586048,46.562,-2.173946,-2.080253,NaN,NaN,NaN,NaN,NaN
AFG-2001,2001,AFG,-,MEA,MNA,IDX,LIC,2.813572e+09,-9.431974,1.516633e+10,...,NaN,110.219341,46.526,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG-2002,2002,AFG,-,MEA,MNA,IDX,LIC,3.825701e+09,28.600001,1.980700e+10,...,NaN,110.534611,46.505,-1.587687,-1.811546,NaN,NaN,NaN,NaN,NaN
AFG-2003,2003,AFG,-,MEA,MNA,IDX,LIC,4.520947e+09,8.832278,2.198200e+10,...,NaN,110.557540,46.497,-1.175768,-1.463108,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2019,2019,ZWE,7,SSF,SSA,IDB,LMC,3.335770e+10,-6.332450,6.361373e+10,...,75.001228,85.447906,65.795,-1.310435,-1.486515,NaN,NaN,3.0,3.428022,3.428022
ZWE-2020,2020,ZWE,7,SSF,SSA,IDB,LMC,3.198033e+10,-7.816951,6.488043e+10,...,NaN,84.384381,64.665,-1.342368,-1.434415,NaN,NaN,3.0,3.642132,3.642132
ZWE-2021,2021,ZWE,7,SSF,SSA,IDB,LMC,4.128767e+10,8.468017,7.625453e+10,...,NaN,83.384953,65.397,-1.290561,-1.386109,NaN,NaN,3.0,4.759522,4.759522


In [55]:
non_float_features_t = list(dataset.columns[dataset.dtypes!=float])

In [7]:
print('Initial Dataset Shape:', dataset.shape)
dataset = dataset[dataset['OECD_RATING'] != '-']
print('Drop Null Target Shape:', dataset.shape)
dataset = selection.filter_missingness(dataset, max_missing_ratio=0.5)
print('Filter Missingness Shape:', dataset.shape)
dataset = dataset[non_float_features_t].join(selection.filter_low_variance(dataset.drop(columns=non_float_features_t)))
print('Filter Low Variance Shape:', dataset.shape)
dataset = dataset[non_float_features_t].join(selection.filter_correlated(dataset.drop(columns=non_float_features_t)))
print('Filter Correlated Shape:', dataset.shape)
dataset = dataset[non_float_features_t].join(selection.select_by_mutual_information(
    dataset.drop(columns=non_float_features_t),
    dataset['OECD_RATING']
))
print('Filter Mutual Information Shape:', dataset.shape)
dataset

Initial Dataset Shape: (5025, 85)
Drop Null Target Shape: (4204, 85)
Filter Missingness Shape: (4204, 76)
Filter Low Variance Shape: (4204, 75)
Filter Correlated Shape: (4204, 56)
Filter Mutual Information Shape: (4204, 56)


,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,GE.EST,NY.GDP.PCAP.CD,DT.DOD.DLXF.CD,...,NE.TRD.GNFS.ZS,FR.INR.RINR,SL.UEM.1524.ZS,BG.GSR.NFSV.GD.ZS,NV.SRV.TOTL.KD.ZG,NY.GNP.MKTP.KD.ZG,NY.GDP.MKTP.KD.ZG,BX.KLT.DINV.WD.GD.ZS,NE.GDI.TOTL.KD.ZG,NV.IND.TOTL.KD.ZG
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-1999,1999,AFG,7,MEA,MNA,IDX,LIC,NaN,NaN,NaN,...,NaN,NaN,10.180,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG-2000,2000,AFG,7,MEA,MNA,IDX,LIC,-2.173946,174.930991,NaN,...,NaN,NaN,10.203,NaN,NaN,NaN,NaN,0.004828,NaN,NaN
AFG-2008,2008,AFG,7,MEA,MNA,IDX,LIC,-1.527795,381.733238,1.994506e+09,...,NaN,12.557960,10.137,20.211726,14.854449,NaN,3.924984,0.455360,NaN,5.741818
AFG-2009,2009,AFG,7,MEA,MNA,IDX,LIC,-1.507752,452.053705,2.106109e+09,...,NaN,17.542929,10.056,20.562137,17.983382,NaN,21.390528,0.451889,NaN,6.107141
AFG-2010,2010,AFG,7,MEA,MNA,IDX,LIC,-1.478316,560.621505,1.975547e+09,...,NaN,11.364094,10.041,20.265308,18.355405,NaN,14.362441,1.203118,NaN,6.270601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2019,2019,ZWE,7,SSF,SSA,IDB,LMC,-1.310435,2184.329239,8.547578e+09,...,46.106544,-76.631576,11.718,4.533750,-3.744152,-5.220787,-6.332450,0.748473,-5.198220,-8.587237
ZWE-2020,2020,ZWE,7,SSF,SSA,IDB,LMC,-1.342368,2059.674454,8.691588e+09,...,40.178406,-80.610262,14.243,3.651449,-12.469129,-9.914347,-7.816951,0.470482,0.832007,-10.039817
ZWE-2021,2021,ZWE,7,SSF,SSA,IDB,LMC,-1.290561,2613.605421,8.951388e+09,...,44.114559,-30.318195,15.208,2.860813,6.864652,7.419142,8.468017,0.575185,31.286247,4.929101


In [8]:
io.save_csv(dataset, config.INTERIM_DATA_DIR / 'feature_selected_dataset.csv', index=True)

# 3 - Feature Pruning (Aligned with Target)

In [43]:
dataset = io.load_csv(config.INTERIM_DATA_DIR / 'feature_selected_dataset.csv', index_col=0)
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,GE.EST,NY.GDP.PCAP.CD,DT.DOD.DLXF.CD,...,NE.TRD.GNFS.ZS,FR.INR.RINR,SL.UEM.1524.ZS,BG.GSR.NFSV.GD.ZS,NV.SRV.TOTL.KD.ZG,NY.GNP.MKTP.KD.ZG,NY.GDP.MKTP.KD.ZG,BX.KLT.DINV.WD.GD.ZS,NE.GDI.TOTL.KD.ZG,NV.IND.TOTL.KD.ZG
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-1999,1999,AFG,7,MEA,MNA,IDX,LIC,NaN,NaN,NaN,...,NaN,NaN,10.180,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG-2000,2000,AFG,7,MEA,MNA,IDX,LIC,-2.173946,174.930991,NaN,...,NaN,NaN,10.203,NaN,NaN,NaN,NaN,0.004828,NaN,NaN
AFG-2008,2008,AFG,7,MEA,MNA,IDX,LIC,-1.527795,381.733238,1.994506e+09,...,NaN,12.557960,10.137,20.211726,14.854449,NaN,3.924984,0.455360,NaN,5.741818
AFG-2009,2009,AFG,7,MEA,MNA,IDX,LIC,-1.507752,452.053705,2.106109e+09,...,NaN,17.542929,10.056,20.562137,17.983382,NaN,21.390528,0.451889,NaN,6.107141
AFG-2010,2010,AFG,7,MEA,MNA,IDX,LIC,-1.478316,560.621505,1.975547e+09,...,NaN,11.364094,10.041,20.265308,18.355405,NaN,14.362441,1.203118,NaN,6.270601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2019,2019,ZWE,7,SSF,SSA,IDB,LMC,-1.310435,2184.329239,8.547578e+09,...,46.106544,-76.631576,11.718,4.533750,-3.744152,-5.220787,-6.332450,0.748473,-5.198220,-8.587237
ZWE-2020,2020,ZWE,7,SSF,SSA,IDB,LMC,-1.342368,2059.674454,8.691588e+09,...,40.178406,-80.610262,14.243,3.651449,-12.469129,-9.914347,-7.816951,0.470482,0.832007,-10.039817
ZWE-2021,2021,ZWE,7,SSF,SSA,IDB,LMC,-1.290561,2613.605421,8.951388e+09,...,44.114559,-30.318195,15.208,2.860813,6.864652,7.419142,8.468017,0.575185,31.286247,4.929101


In [44]:
dataset = dataset[(dataset.isna().sum(axis=1) / len(dataset.columns)) <= 0.5]
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,GE.EST,NY.GDP.PCAP.CD,DT.DOD.DLXF.CD,...,NE.TRD.GNFS.ZS,FR.INR.RINR,SL.UEM.1524.ZS,BG.GSR.NFSV.GD.ZS,NV.SRV.TOTL.KD.ZG,NY.GNP.MKTP.KD.ZG,NY.GDP.MKTP.KD.ZG,BX.KLT.DINV.WD.GD.ZS,NE.GDI.TOTL.KD.ZG,NV.IND.TOTL.KD.ZG
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-2008,2008,AFG,7,MEA,MNA,IDX,LIC,-1.527795,381.733238,1.994506e+09,...,NaN,12.557960,10.137,20.211726,14.854449,NaN,3.924984,0.455360,NaN,5.741818
AFG-2009,2009,AFG,7,MEA,MNA,IDX,LIC,-1.507752,452.053705,2.106109e+09,...,NaN,17.542929,10.056,20.562137,17.983382,NaN,21.390528,0.451889,NaN,6.107141
AFG-2010,2010,AFG,7,MEA,MNA,IDX,LIC,-1.478316,560.621505,1.975547e+09,...,NaN,11.364094,10.041,20.265308,18.355405,NaN,14.362441,1.203118,NaN,6.270601
AFG-2011,2011,AFG,7,MEA,MNA,IDX,LIC,-1.474100,606.694676,2.032464e+09,...,NaN,-1.241506,10.044,23.334223,9.790686,NaN,0.426355,0.293025,NaN,9.807670
AFG-2012,2012,AFG,7,MEA,MNA,IDX,LIC,-1.375535,651.417134,2.079786e+09,...,NaN,7.174387,10.071,18.250447,13.657340,NaN,12.752287,0.285441,NaN,6.394071
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2019,2019,ZWE,7,SSF,SSA,IDB,LMC,-1.310435,2184.329239,8.547578e+09,...,46.106544,-76.631576,11.718,4.533750,-3.744152,-5.220787,-6.332450,0.748473,-5.198220,-8.587237
ZWE-2020,2020,ZWE,7,SSF,SSA,IDB,LMC,-1.342368,2059.674454,8.691588e+09,...,40.178406,-80.610262,14.243,3.651449,-12.469129,-9.914347,-7.816951,0.470482,0.832007,-10.039817
ZWE-2021,2021,ZWE,7,SSF,SSA,IDB,LMC,-1.290561,2613.605421,8.951388e+09,...,44.114559,-30.318195,15.208,2.860813,6.864652,7.419142,8.468017,0.575185,31.286247,4.929101


In [45]:
non_float_features = []
for c in non_float_features_t:
    if c != 'OECD_RATING':
        non_float_features.append(c)

In [47]:
X = dataset.drop(columns=['OECD_RATING']) # Drop Target
X = X.drop(columns=['ISO3_COUNTRY_CODE']) # Drop Country ID (maybe YEAR)
y = dataset['OECD_RATING']

In [48]:
io.save_csv(X, config.PROCESSED_DATA_DIR / 'X.csv', index=True)

In [49]:
io.save_csv(y, config.PROCESSED_DATA_DIR / 'y.csv', index=True)